# Chapter 5: Applying Financial Metrics Using SQL

## 1. Financial Data in a SQL Context


**Why SQL Matters in Financial Analysis**

In real-world environments, financial data
is typically stored in relational databases.

Financial analysts use SQL to:
- Extract financial statements
- Aggregate metrics
- Perform trend and variance analysis
directly within data warehouses.


**SQL vs Python in Financial Analysis**

Python is often used for advanced analysis
and visualization.

SQL is used for:
- Data extraction
- Data validation
- Metric calculation at scale

Both skills are essential for
modern financial analysts.

**Data Structure Overview**

In this chapter, we will assume
financial data is stored in a table
with yearly financial information.

Each row represents a financial period,
and each column represents a financial metric.

In the next section, we will design
and create a financial table
using SQL.


> **Note:** No external SQL installation is required — we use SQLite directly within Jupyter for a fully reproducible workflow.


In [1]:
import sqlite3
import pandas as pd


In [2]:
# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()


## 2. Creating Financial Tables


In [3]:
create_table_query = """
CREATE TABLE financials (
    year INTEGER,
    revenue REAL,
    cogs REAL,
    operating_expenses REAL,
    interest_expense REAL,
    tax_expense REAL,
    total_assets REAL,
    total_liabilities REAL,
    equity REAL
);
"""
cursor.execute(create_table_query)
conn.commit()


##### Insert Sample Data

In [4]:
insert_data_query = """
INSERT INTO financials VALUES
(2019, 500, 300, 120, 20, 15, 800, 400, 400),
(2020, 520, 310, 125, 22, 14, 820, 420, 400),
(2021, 560, 330, 130, 21, 16, 850, 440, 410),
(2022, 600, 350, 135, 20, 18, 900, 460, 440),
(2023, 650, 380, 140, 18, 20, 950, 480, 470);
"""
cursor.execute(insert_data_query)
conn.commit()


##### Validate Table

In [5]:
pd.read_sql("SELECT * FROM financials;", conn)


,year,revenue,cogs,operating_expenses,interest_expense,tax_expense,total_assets,total_liabilities,equity
0,2019,500.0,300.0,120.0,20.0,15.0,800.0,400.0,400.0
1,2020,520.0,310.0,125.0,22.0,14.0,820.0,420.0,400.0
2,2021,560.0,330.0,130.0,21.0,16.0,850.0,440.0,410.0
3,2022,600.0,350.0,135.0,20.0,18.0,900.0,460.0,440.0
4,2023,650.0,380.0,140.0,18.0,20.0,950.0,480.0,470.0


**Why This Matters**

The financial table structure closely resembles
how financial data is stored in enterprise databases.

Using SQL within Jupyter allows analysts
to document logic while maintaining reproducibility.


The financial table is now created and populated.

In the next section, we will write SQL queries
to calculate key financial metrics.


## 3. Writing SQL for Financial Metrics


##### Gross Profit & Gross Margin

In [6]:
gross_profit_query = """
SELECT
    year,
    revenue,
    cogs,
    revenue - cogs AS gross_profit,
    ROUND((revenue - cogs) * 1.0 / revenue, 2) AS gross_margin
FROM financials;
"""
pd.read_sql(gross_profit_query, conn)


,year,revenue,cogs,gross_profit,gross_margin
0,2019,500.0,300.0,200.0,0.40
1,2020,520.0,310.0,210.0,0.40
2,2021,560.0,330.0,230.0,0.41
3,2022,600.0,350.0,250.0,0.42
4,2023,650.0,380.0,270.0,0.42


##### Operating Profit & Operating Margin

In [7]:
operating_profit_query = """
SELECT
    year,
    revenue,
    (revenue - cogs - operating_expenses) AS operating_profit,
    ROUND(
        (revenue - cogs - operating_expenses) * 1.0 / revenue, 2
    ) AS operating_margin
FROM financials;
"""
pd.read_sql(operating_profit_query, conn)


,year,revenue,operating_profit,operating_margin
0,2019,500.0,80.0,0.16
1,2020,520.0,85.0,0.16
2,2021,560.0,100.0,0.18
3,2022,600.0,115.0,0.19
4,2023,650.0,130.0,0.20


##### Net Profit & Net Margin

In [8]:
net_profit_query = """
SELECT
    year,
    revenue,
    (revenue - cogs - operating_expenses
     - interest_expense - tax_expense) AS net_profit,
    ROUND(
        (revenue - cogs - operating_expenses
         - interest_expense - tax_expense) * 1.0 / revenue, 2
    ) AS net_margin
FROM financials;
"""
pd.read_sql(net_profit_query, conn)


,year,revenue,net_profit,net_margin
0,2019,500.0,45.0,0.09
1,2020,520.0,49.0,0.09
2,2021,560.0,63.0,0.11
3,2022,600.0,77.0,0.13
4,2023,650.0,92.0,0.14


##### Liquidity & Leverage Metrics

In [9]:
liquidity_leverage_query = """
SELECT
    year,
    ROUND(total_assets * 1.0 / total_liabilities, 2) AS current_ratio,
    ROUND(total_liabilities * 1.0 / equity, 2) AS debt_to_equity
FROM financials;
"""
pd.read_sql(liquidity_leverage_query, conn)


,year,current_ratio,debt_to_equity
0,2019,2.00,1.00
1,2020,1.95,1.05
2,2021,1.93,1.07
3,2022,1.96,1.05
4,2023,1.98,1.02


**Why SQL Matters**

Using SQL for financial metrics allows analysts
to compute results directly at the data source.

This improves performance, consistency,
and scalability in enterprise environments.


Core financial metrics have now been
calculated using SQL.

In the next section, we will analyze
trends and variances using SQL queries.


## 4. Trend & Variance Analysis Using SQL


##### Year-over-Year Revenue Growth

In [10]:
revenue_trend_query = """
SELECT
    year,
    revenue,
    revenue - LAG(revenue) OVER (ORDER BY year) AS revenue_change,
    ROUND(
        (revenue - LAG(revenue) OVER (ORDER BY year)) * 1.0
        / LAG(revenue) OVER (ORDER BY year),
        2
    ) AS revenue_growth_rate
FROM financials;
"""
pd.read_sql(revenue_trend_query, conn)


,year,revenue,revenue_change,revenue_growth_rate
0,2019,500.0,NaN,NaN
1,2020,520.0,20.0,0.04
2,2021,560.0,40.0,0.08
3,2022,600.0,40.0,0.07
4,2023,650.0,50.0,0.08


##### Margin Trend Analysis

In [11]:
margin_trend_query = """
SELECT
    year,
    ROUND((revenue - cogs) * 1.0 / revenue, 2) AS gross_margin,
    ROUND(
        ((revenue - cogs) * 1.0 / revenue)
        - LAG((revenue - cogs) * 1.0 / revenue)
          OVER (ORDER BY year),
        2
    ) AS gross_margin_change
FROM financials;
"""
pd.read_sql(margin_trend_query, conn)


,year,gross_margin,gross_margin_change
0,2019,0.40,NaN
1,2020,0.40,0.00
2,2021,0.41,0.01
3,2022,0.42,0.01
4,2023,0.42,-0.00


##### Expense Variance Analysis

In [12]:
expense_variance_query = """
SELECT
    year,
    operating_expenses,
    operating_expenses
    - LAG(operating_expenses) OVER (ORDER BY year)
      AS expense_variance
FROM financials;
"""
pd.read_sql(expense_variance_query, conn)


,year,operating_expenses,expense_variance
0,2019,120.0,NaN
1,2020,125.0,5.0
2,2021,130.0,5.0
3,2022,135.0,5.0
4,2023,140.0,5.0


Trend and variance analysis highlight
changes in financial performance over time.

Year-over-year comparisons help identify:
- Growth momentum
- Cost control effectiveness
- Emerging financial risks


SQL window functions enable analysts
to perform time-series analysis efficiently.

In the next section, we interpret these
results from a business perspective.


## 5. Interpreting Results Like an Analyst


**Why Interpretation Matters**

Writing SQL queries and calculating metrics is only the first step.

The real value of financial analysis comes from interpreting query outputs and translating database results into business insights that support decision-making.

SQL allows analysts to work directly on production data, making interpretation even more critical.

**Revenue & Growth Interpretation**

Year-over-year revenue trends derived using SQL window functions reveal how the business is evolving over time.

Consistent revenue growth suggests:

- Strong demand for products or services

- Effective pricing and sales strategies

- Healthy market positioning

Sudden slowdowns or declines may indicate market pressure, competitive threats, or execution challenges that require deeper investigation.

**Profitability Interpretation**

Margin calculations performed in SQL highlight how efficiently the company converts revenue into profit.

Improving gross or operating margins suggest:

- Better cost control

- Improved operational efficiency

- Increased pricing power

Declining margins may signal rising input costs, inefficiencies, or discounting strategies that could impact long-term profitability.

**Expense & Variance Interpretation**

Expense variance analysis using SQL helps identify where costs are changing over time.

Positive variance (controlled or declining expenses) suggests:

- Operational discipline

- Effective budgeting

- Scalable cost structures

Negative variance (rising expenses) may require management attention to understand cost drivers and prevent margin erosion.

**Liquidity & Risk Perspective**

SQL-based analysis of leverage and coverage metrics provides insight into financial stability.

Stable leverage and strong coverage ratios indicate:

- Controlled use of debt

- Adequate ability to service obligations

- Lower financial risk

Sharp increases in leverage or weakening coverage metrics may raise concerns about solvency and future cash flow pressure.

**Connecting SQL Metrics to Business Decisions**

From an FP&A and management reporting perspective, SQL-driven insights support:

- Investment and expansion decisions

- Cost optimization initiatives

- Risk monitoring and financial control

- Performance tracking against strategic goals

Because SQL operates directly on enterprise data, these insights are often used in dashboards, reports, and executive briefings.

**How Analysts Think When Reviewing SQL Outputs**

Good financial analysts do not stop at query results.

They ask:

- Why did this metric change from last period?

- Is this trend structural or temporary?

- Which business drivers explain this movement?

- What actions should management consider?

Interpretation transforms SQL queries from technical exercises into strategic decision-making tools.

**Section Takeaway**

SQL is not just a data extraction tool — it is a core analytical language for financial professionals.

Strong analysts combine accurate SQL queries with clear interpretation to deliver insights that influence real business outcomes.

In the next section, we summarize the key learnings from this chapter and transition toward advanced financial analysis applications.

##  Chapter Summary & Transition



### Chapter Summary

In this chapter, we applied financial analysis directly within a SQL environment using real-world querying techniques.

We demonstrated how structured SQL queries can be used to calculate key financial metrics, analyze trends over time, and evaluate business performance directly from enterprise databases.

This chapter reinforced the idea that SQL is not just a data retrieval tool, but a core analytical language for financial professionals working with large-scale financial data.


### Key Skills Demonstrated

This chapter demonstrated:

- Financial metric calculation using SQL  
- Trend analysis with SQL window functions  
- Variance and performance analysis directly in databases  
- Analytical interpretation of SQL query results  
- Translating database outputs into business insights  

These skills reflect how financial analysts operate in production data environments.


### Analyst Takeaway

Strong financial analysis is not defined by the tool used, but by the analyst’s ability to extract meaning from data.

SQL enables analysts to work closer to the source of truth, making interpretation, context, and judgment even more important.

Clear thinking, structured queries, and disciplined interpretation lead to better financial decisions.


In the next chapter, we will shift from querying financial data to **financial planning and performance analysis**.

We will focus on FP&A-style analysis, including budgeting, forecasting, and performance evaluation, building on the analytical foundations developed in this chapter.

This transition reflects how analysts move from historical analysis toward forward-looking financial decision-making.
